### BLS/OEWS data provenance

The occupational employment and wage data used in this analysis were obtained
from the U.S. Bureau of Labor Statistics Occupational Employment and Wage
Statistics (OEWS) national data tables.

The analysis uses the May national OEWS releases for:

- Pre-period: 2017, 2018, and 2019
- Post-period: 2023, 2024, and 2025

The original national XLSX files were downloaded from the official BLS OEWS
tables archive and preserved locally under:

`08_Labor_Outcomes/BLS/Input/`

All subsequent processing reads from these preserved local copies rather than
retrieving files dynamically from the BLS website.

**Official source:** [BLS Occupational Employment and Wage Statistics (OEWS) Tables](https://www.bls.gov/oes/tables.htm)



# Occupational Employment and Wage Outcome Construction

## Analytical objective

Construct comparable SOC6-level measures of occupational employment and wage trajectories for the RQ1 outcome models.

## Measurement design

The analysis uses national Occupational Employment and Wage Statistics (OEWS) estimates for two periods:

* **Pre-period:** May 2017–2019
* **Post-period:** May 2023–2025

Each occupation’s annual estimates are averaged within the pre- and post-periods. The primary trajectory measure is the annualized log change between the two period averages:

$$
\Delta Y_o
=
\frac{100}{6}
\left[
\ln(\overline{Y}_{o,post})
-
\ln(\overline{Y}_{o,pre})
\right]
$$

where \(Y\) represents occupational employment or real median wage, and six is the interval between the period midpoints, 2018 and 2024.

Employment is measured using national OEWS wage-and-salary employment (`TOT_EMP`). Wages are converted to May 2025 dollars using the CPI-U before calculating wage change. The final choice between hourly and annual median wage is determined from the cross-year coverage audit; the two measures are not mixed.

## Comparability rule

The frozen configuration–exposure universe contains 604 SOC6 occupations. Labor outcomes are constructed only where the historical and current SOC definitions are sufficiently comparable. Employment and wage models may therefore contain different numbers of occupations because employment counts can sometimes be aggregated across predecessor codes, whereas median wages cannot be reconstructed from component medians.

These measures describe occupational trajectories. They do not, by themselves, identify causal effects of GenAI.


In [ ]:
# ---------------------------------------------------------------
# Step 1 — Verify BLS/OEWS source files
# Purpose: confirm all prespecified annual source files are present
# ---------------------------------------------------------------
import pandas as pd
from pathlib import Path
import numpy as np
import re

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "GenAI-task-affordance-labor-outcomes"
)

bls_dir  = (
    BASE_DIR
    / "08_Labor_Outcomes"
     / "Input"
    / "BLS"
)

print(bls_dir)

pre_years = [2017, 2018, 2019]
post_years = [2023, 2024, 2025]
analysis_years = pre_years + post_years

bls_files = {
    year: bls_dir / f"national_M{year}_dl.xlsx"
    for year in analysis_years
}

for year, path in bls_files.items():
    print(year, ":", path.exists(), "-", path.name)


In [ ]:
# ------------------------------------------------------------------
# Step 2 — Load frozen RQ1 occupation universe
# Purpose: define the occupations retained for BLS outcome construction
# ------------------------------------------------------------------

RQ1_FILE = (
    BASE_DIR
    / "07_GenAI_Exposure"
    / "Output"
    / "Occupational_Configuration_Exposure_v1.csv"
)

rq1_sample = pd.read_csv(
    RQ1_FILE,
    dtype={"soc6": str}
)

target_socs = sorted(
    rq1_sample["soc6"]
    .dropna()
    .unique()
)

print("Target occupations:", len(target_socs))
print("Unique SOC6:", rq1_sample["soc6"].nunique())

In [ ]:
# ------------------------------------------------------------------
# Step 3 — Audit raw OEWS files
# Purpose:
#   1. verify workbook structure and required fields
#   2. preserve suppression information
#   3. inspect unharmonized SOC coverage
#   4. retain the national 00-0000 employment denominator
# ------------------------------------------------------------------

import hashlib


def normalized_name(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def find_column(columns, candidates):
    column_map = {
        normalized_name(column): column
        for column in columns
    }

    for candidate in candidates:
        match = column_map.get(normalized_name(candidate))
        if match is not None:
            return match

    return None


def normalize_soc_code(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"(\d{2}-\d{4})", str(value))
    return match.group(1) if match else np.nan


def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


required_fields = {
    "soc_code": ["OCC_CODE", "SOC_CODE"],
    "occupation_title": ["OCC_TITLE", "OCCUPATION_TITLE"],
    "occupation_group": ["OCC_GROUP", "O_GROUP"],
    "employment": ["TOT_EMP"],
    "hourly_mean": ["H_MEAN"],
    "hourly_median": ["H_MEDIAN"],
    "annual_mean": ["A_MEAN"],
    "annual_median": ["A_MEDIAN"],
}

file_rows = []
field_rows = []

for year, path in bls_files.items():

    workbook = pd.ExcelFile(path)

    sheet = next(
        (
            name for name in workbook.sheet_names
            if normalized_name(name) == "NATIONALDL"
        ),
        workbook.sheet_names[0]
    )

    raw = pd.read_excel(
        path,
        sheet_name=sheet,
        dtype=str
    )

    raw.columns = [
        str(column).strip()
        for column in raw.columns
    ]

    detected = {
        field: find_column(raw.columns, candidates)
        for field, candidates in required_fields.items()
    }

    soc_col = detected["soc_code"]
    group_col = detected["occupation_group"]
    employment_col = detected["employment"]

    if soc_col is None:
        raise KeyError(f"{year}: occupation-code field not found")

    raw["_soc_code_audit"] = raw[soc_col].apply(normalize_soc_code)

    if group_col is not None:
        detailed_mask = (
            raw[group_col]
            .fillna("")
            .str.strip()
            .str.lower()
            .eq("detailed")
        )
    else:
        detailed_mask = pd.Series(False, index=raw.index)

    detailed = raw.loc[detailed_mask].copy()

    target_exact = set(
        detailed["_soc_code_audit"].dropna()
    ).intersection(target_socs)

    total_row = raw.loc[
        raw["_soc_code_audit"].eq("00-0000")
    ]

    total_employment_raw = np.nan

    if employment_col is not None and not total_row.empty:
        total_employment_raw = total_row.iloc[0][employment_col]

    file_rows.append({
        "year": year,
        "filename": path.name,
        "sheet": sheet,
        "file_size_mb": round(path.stat().st_size / 1_000_000, 2),
        "sha256": file_sha256(path),
        "raw_rows": len(raw),
        "raw_columns": len(raw.columns) - 1,
        "detailed_rows": int(detailed_mask.sum()),
        "detailed_unique_soc": detailed["_soc_code_audit"].nunique(),
        "exact_target_soc_before_mapping": len(target_exact),
        "target_soc_not_exactly_present": len(target_socs) - len(target_exact),
        "total_00_0000_present": not total_row.empty,
        "total_employment_raw": total_employment_raw,
    })

    for field, column in detected.items():

        if column is None:
            field_rows.append({
                "year": year,
                "field": field,
                "source_column": None,
                "numeric_detailed": 0,
                "blank_detailed": np.nan,
                "nonnumeric_detailed": np.nan,
                "nonnumeric_tokens": None,
            })
            continue

        values = (
            detailed[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        numeric = pd.to_numeric(
            values.str.replace(",", "", regex=False),
            errors="coerce"
        )

        blank = values.eq("")
        nonnumeric = values.ne("") & numeric.isna()

        tokens = (
            values.loc[nonnumeric]
            .value_counts()
            .head(10)
            .to_dict()
        )

        field_rows.append({
            "year": year,
            "field": field,
            "source_column": column,
            "numeric_detailed": int(numeric.notna().sum()),
            "blank_detailed": int(blank.sum()),
            "nonnumeric_detailed": int(nonnumeric.sum()),
            "nonnumeric_tokens": tokens,
        })


oews_file_inventory = (
    pd.DataFrame(file_rows)
    .sort_values("year")
    .reset_index(drop=True)
)

oews_field_inventory = (
    pd.DataFrame(field_rows)
    .sort_values(["year", "field"])
    .reset_index(drop=True)
)

display(oews_file_inventory)

display(
    oews_field_inventory[
        oews_field_inventory["field"].isin([
            "employment",
            "hourly_mean",
            "hourly_median",
            "annual_mean",
            "annual_median",
        ])
    ]
)

In [ ]:
# ------------------------------------------------------------------
# One-time extraction — BLS retrospective MB3 workbooks
# ------------------------------------------------------------------

from pathlib import Path
import zipfile
import shutil

MB3_DIR = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Input"
    / "BLS_MB3"
)

for year in [2017, 2018, 2019]:

    zip_path = MB3_DIR / f"oes_mb3_{year}a.zip"
    xlsx_path = MB3_DIR / f"oes_mb3_{year}a.xlsx"

    if xlsx_path.exists():
        print(year, ": already extracted")
        continue

    with zipfile.ZipFile(zip_path) as archive:

        member = next(
            name for name in archive.namelist()
            if name.lower().endswith(".xlsx")
        )

        with archive.open(member) as source:
            with open(xlsx_path, "wb") as destination:
                shutil.copyfileobj(source, destination)

    print(
        year,
        ": extracted —",
        xlsx_path.name,
        f"({xlsx_path.stat().st_size / 1_000_000:.1f} MB)"
    )

In [ ]:
# ------------------------------------------------------------------
# Step 4 — Register final OEWS source files
# Purpose: use retrospective MB3 files for the pre-period and
#          official MB3 releases for the post-period
# ------------------------------------------------------------------

BLS_DIR = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Input"
    / "BLS"
)

MB3_DIR = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Input"
    / "BLS_MB3"
)

bls_files = {
    2017: MB3_DIR / "oes_mb3_2017a.xlsx",
    2018: MB3_DIR / "oes_mb3_2018a.xlsx",
    2019: MB3_DIR / "oes_mb3_2019a.xlsx",
    2023: BLS_DIR / "national_M2023_dl.xlsx",
    2024: BLS_DIR / "national_M2024_dl.xlsx",
    2025: BLS_DIR / "national_M2025_dl.xlsx",
}

source_type = {
    2017: "Retrospective BLS MB3 research estimate",
    2018: "Retrospective BLS MB3 research estimate",
    2019: "Retrospective BLS MB3 research estimate",
    2023: "Official published MB3 estimate",
    2024: "Official published MB3 estimate",
    2025: "Official published MB3 estimate",
}

for year, path in bls_files.items():

    if not path.exists():
        raise FileNotFoundError(path)

    workbook = pd.ExcelFile(path)

    print(
        year,
        "|",
        source_type[year],
        "|",
        path.name,
        "|",
        round(path.stat().st_size / 1_000_000, 1),
        "MB",
        "| sheet:",
        workbook.sheet_names[0]
    )

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Extract national cross-industry occupation frames
# Purpose: recompute direct SOC coverage using the final MB3 sources
# Extract national cross-industry records from the MB3 all-data workbooks.
# Recalculate direct SOC matches.
# Identify the historical codes corresponding to the unmatched current SOCs.
# Classify the mappings as one-to-one, many-to-one, split, or noncomparable.
# ------------------------------------------------------------------

wanted_fields = {
    "AREA", "NAICS", "I_GROUP", "OWN_CODE",
    "OCC_CODE", "SOC_CODE", "OCC_TITLE",
    "O_GROUP", "OCC_GROUP",
    "TOT_EMP", "H_MEAN", "H_MEDIAN",
    "A_MEAN", "A_MEDIAN"
}

final_national_frames = {}
coverage_rows = []

for year, path in bls_files.items():

    raw = pd.read_excel(
        path,
        sheet_name=0,
        dtype=str,
        usecols=lambda column: (
            normalized_name(column)
            in {normalized_name(x) for x in wanted_fields}
        )
    )

    raw.columns = [
        str(column).strip()
        for column in raw.columns
    ]

    area_col = find_column(raw.columns, ["AREA"])
    naics_col = find_column(raw.columns, ["NAICS"])
    industry_group_col = find_column(raw.columns, ["I_GROUP"])
    occupation_group_col = find_column(
        raw.columns,
        ["O_GROUP", "OCC_GROUP"]
    )
    soc_col = find_column(
        raw.columns,
        ["OCC_CODE", "SOC_CODE"]
    )

    # National records
    if area_col is not None:
        area_numeric = pd.to_numeric(
            raw[area_col],
            errors="coerce"
        )
        raw = raw.loc[area_numeric.eq(99)].copy()

    # Cross-industry records
    if naics_col is not None:
        naics = (
            raw[naics_col]
            .fillna("")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
            .str.zfill(6)
        )
        raw = raw.loc[naics.eq("000000")].copy()

    if industry_group_col is not None:
        industry_group = (
            raw[industry_group_col]
            .fillna("")
            .str.strip()
            .str.lower()
        )
        raw = raw.loc[
            industry_group.eq("cross-industry")
        ].copy()

    # Detailed occupations
    occupation_group = (
        raw[occupation_group_col]
        .fillna("")
        .str.strip()
        .str.lower()
    )

    detailed = raw.loc[
        occupation_group.eq("detailed")
    ].copy()

    detailed["soc6_source"] = (
        detailed[soc_col]
        .apply(normalize_soc_code)
    )

    final_national_frames[year] = detailed

    source_socs = set(
        detailed["soc6_source"].dropna()
    )

    direct_matches = set(target_socs).intersection(source_socs)

    coverage_rows.append({
        "year": year,
        "detailed_soc": len(source_socs),
        "direct_target_matches": len(direct_matches),
        "target_not_directly_present": (
            len(target_socs) - len(direct_matches)
        ),
    })

final_source_coverage = pd.DataFrame(coverage_rows)

display(final_source_coverage)

In [ ]:
# ------------------------------------------------------------------
# Step 6 — Cache filtered national detailed records
# Purpose: avoid repeatedly parsing the large MB3 workbooks
# ------------------------------------------------------------------

CACHE_DIR = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Output"
    / "Source_Cache"
)

CACHE_DIR.mkdir(parents=True, exist_ok=True)

for year, frame in final_national_frames.items():

    group_col = find_column(
        frame.columns,
        ["O_GROUP", "OCC_GROUP"]
    )

    detailed = frame[
        frame[group_col]
        .fillna("")
        .str.strip()
        .str.lower()
        .eq("detailed")
    ].copy()

    cache_file = (
        CACHE_DIR
        / f"OEWS_National_Detailed_{year}.csv"
    )

    detailed.to_csv(cache_file, index=False)

    print(
        year,
        ":",
        len(detailed),
        "detailed occupations saved"
    )

In [ ]:
# ------------------------------------------------------------------
# Step 7 — Identify SOC6 codes requiring historical harmonization
# Purpose: isolate the exact 2017–2019 correspondence problem
# ------------------------------------------------------------------

soc_sets = {}
cached_frames = {}

for year in analysis_years:

    cache_file = (
        CACHE_DIR
        / f"OEWS_National_Detailed_{year}.csv"
    )

    frame = pd.read_csv(cache_file, dtype=str)

    soc_col = find_column(
        frame.columns,
        ["OCC_CODE", "SOC_CODE"]
    )

    frame["soc6_source"] = (
        frame[soc_col]
        .apply(normalize_soc_code)
    )

    cached_frames[year] = frame
    soc_sets[year] = set(
        frame["soc6_source"].dropna()
    )


# Current occupation titles from the 2025 file
title_col_2025 = find_column(
    cached_frames[2025].columns,
    ["OCC_TITLE", "OCCUPATION_TITLE"]
)

current_titles = (
    cached_frames[2025]
    .set_index("soc6_source")[title_col_2025]
    .to_dict()
)


soc_coverage_audit = pd.DataFrame({
    "soc6": target_socs
})

soc_coverage_audit["occupation_title_2025"] = (
    soc_coverage_audit["soc6"]
    .map(current_titles)
)

for year in analysis_years:
    soc_coverage_audit[f"present_{year}"] = (
        soc_coverage_audit["soc6"]
        .isin(soc_sets[year])
    )


historical_pattern_summary = (
    soc_coverage_audit
    .groupby(
        ["present_2017", "present_2018", "present_2019"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "occupation_count"})
)

historical_unmatched = (
    soc_coverage_audit[
        ~(
            soc_coverage_audit["present_2017"]
            & soc_coverage_audit["present_2018"]
            & soc_coverage_audit["present_2019"]
        )
    ]
    .reset_index(drop=True)
)

display(historical_pattern_summary)

print(
    "Occupations requiring historical review:",
    len(historical_unmatched)
)

display(historical_unmatched)

In [ ]:
CROSSWALK_DIR = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Input"
    / "SOC_Crosswalk"
)

correspondence_files = {
    "2010-to-2018 crosswalk":
        CROSSWALK_DIR / "soc_2010_to_2018_crosswalk.xlsx",

    "2019 hybrid structure":
        CROSSWALK_DIR / "oes_2019_hybrid_structure.xlsx",
}

for label, file_path in correspondence_files.items():
    workbook = pd.ExcelFile(file_path)

    print(f"\n{label}")
    print("File:", file_path.name)
    print("Sheets:", workbook.sheet_names)

    preview = pd.read_excel(
        file_path,
        sheet_name=workbook.sheet_names[0],
        header=None,
        nrows=12
    )

    display(preview)

In [ ]:
import re
import pandas as pd

def extract_soc(value):
    """Extract a six-digit SOC code while retaining split/merge notation separately."""
    match = re.search(r"\d{2}-\d{4}", str(value))
    return match.group(0) if match else pd.NA


# ------------------------------------------------------------
# Official 2010 → 2018 SOC crosswalk
# ------------------------------------------------------------
soc_crosswalk = pd.read_excel(
    CROSSWALK_DIR / "soc_2010_to_2018_crosswalk.xlsx",
    sheet_name="Sorted by 2010",
    header=8,
    dtype=str
)

soc_crosswalk.columns = soc_crosswalk.columns.str.strip()

soc_crosswalk = soc_crosswalk.rename(columns={
    "2010 SOC Code": "soc2010_raw",
    "2010 SOC Title": "title2010",
    "2018 SOC Code": "soc2018_raw",
    "2018 SOC Title": "title2018"
})

soc_crosswalk["soc2010"] = soc_crosswalk["soc2010_raw"].map(extract_soc)
soc_crosswalk["soc2018"] = soc_crosswalk["soc2018_raw"].map(extract_soc)

soc_crosswalk = soc_crosswalk.dropna(
    subset=["soc2010", "soc2018"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Official 2019 OEWS hybrid structure
# ------------------------------------------------------------
hybrid_2019 = pd.read_excel(
    CROSSWALK_DIR / "oes_2019_hybrid_structure.xlsx",
    sheet_name="OES2019 Hybrid",
    header=5,
    dtype=str
)

hybrid_2019.columns = hybrid_2019.columns.str.strip()

hybrid_2019 = hybrid_2019.rename(columns={
    "OES 2019 Estimates Code": "oes2019_raw",
    "OES 2019 Estimates Title": "oes2019_title",
    "2018 SOC Code": "soc2018_raw",
    "2018 SOC Title": "soc2018_title",
    "OES 2018 Estimates Code": "oes2018_raw",
    "OES 2018 Estimates Title": "oes2018_title",
    "2010 SOC Code": "soc2010_raw",
    "2010 SOC Title": "soc2010_title",
    "NOTES": "notes"
})

for raw_column, clean_column in [
    ("oes2019_raw", "oes2019"),
    ("soc2018_raw", "soc2018"),
    ("oes2018_raw", "oes2018"),
    ("soc2010_raw", "soc2010")
]:
    hybrid_2019[clean_column] = hybrid_2019[raw_column].map(extract_soc)

hybrid_2019 = hybrid_2019.dropna(
    subset=["oes2019"]
).reset_index(drop=True)


print("2010→2018 crosswalk rows:", len(soc_crosswalk))
print("Unique 2010 SOC:", soc_crosswalk["soc2010"].nunique())
print("Unique 2018 SOC:", soc_crosswalk["soc2018"].nunique())

print("\n2019 hybrid rows:", len(hybrid_2019))
print("Unique 2019 OEWS codes:", hybrid_2019["oes2019"].nunique())
print("Unique represented 2018 SOC:", hybrid_2019["soc2018"].nunique())

display(soc_crosswalk.head())
display(hybrid_2019.head())

In [ ]:
# ------------------------------------------------------------
# Audit official SOC correspondence topology for all 604 SOC6
# ------------------------------------------------------------

def build_relation_maps(df, source_col, target_col):
    pairs = (
        df[[source_col, target_col]]
        .dropna()
        .drop_duplicates()
    )

    target_to_sources = (
        pairs.groupby(target_col)[source_col]
        .apply(lambda x: sorted(x.unique()))
        .to_dict()
    )

    source_to_targets = (
        pairs.groupby(source_col)[target_col]
        .apply(lambda x: sorted(x.unique()))
        .to_dict()
    )

    return target_to_sources, source_to_targets


def classify_topology(target, target_to_sources, source_to_targets):
    sources = target_to_sources.get(target, [])

    if not sources:
        return "no_official_mapping"

    targets_per_source = [
        len(source_to_targets.get(source, []))
        for source in sources
    ]

    if len(sources) == 1 and targets_per_source[0] == 1:
        return "one_to_one"

    if len(sources) > 1 and all(n == 1 for n in targets_per_source):
        return "multiple_sources_to_one_target"

    if len(sources) == 1 and targets_per_source[0] > 1:
        return "one_source_to_multiple_targets"

    return "complex_many_to_many"


# 2017–2018: 2010 SOC source codes → current 2018 SOC targets
cw_target_to_source, cw_source_to_target = build_relation_maps(
    soc_crosswalk,
    source_col="soc2010",
    target_col="soc2018"
)

# 2019: published hybrid OEWS codes → represented 2018 SOC targets
hybrid_target_to_source, hybrid_source_to_target = build_relation_maps(
    hybrid_2019,
    source_col="oes2019",
    target_col="soc2018"
)


correspondence_rows = []

for soc6 in sorted(target_socs):

    pre_sources = cw_target_to_source.get(soc6, [])
    hybrid_sources = hybrid_target_to_source.get(soc6, [])

    correspondence_rows.append({
        "soc6": soc6,

        "soc2010_topology": classify_topology(
            soc6, cw_target_to_source, cw_source_to_target
        ),
        "soc2010_source_codes": " | ".join(pre_sources),
        "soc2010_source_count": len(pre_sources),
        "sources_present_2017": sum(
            code in soc_sets[2017] for code in pre_sources
        ),
        "sources_present_2018": sum(
            code in soc_sets[2018] for code in pre_sources
        ),

        "hybrid2019_topology": classify_topology(
            soc6, hybrid_target_to_source, hybrid_source_to_target
        ),
        "hybrid2019_source_codes": " | ".join(hybrid_sources),
        "hybrid2019_source_count": len(hybrid_sources),
        "sources_present_2019": sum(
            code in soc_sets[2019] for code in hybrid_sources
        )
    })


soc_correspondence_audit = pd.DataFrame(correspondence_rows)

assert len(soc_correspondence_audit) == 604
assert soc_correspondence_audit["soc6"].is_unique

print("2010→2018 topology:")
display(
    soc_correspondence_audit["soc2010_topology"]
    .value_counts(dropna=False)
    .rename_axis("topology")
    .reset_index(name="occupation_count")
)

print("\n2019 hybrid topology:")
display(
    soc_correspondence_audit["hybrid2019_topology"]
    .value_counts(dropna=False)
    .rename_axis("topology")
    .reset_index(name="occupation_count")
)

display(soc_correspondence_audit.head())

In [ ]:
# ------------------------------------------------------------
# Structural eligibility before checking outcome missingness
# ------------------------------------------------------------

aggregable_topologies = {
    "one_to_one",
    "multiple_sources_to_one_target"
}

def all_sources_present(row, year):
    source_count = row[f"{'soc2010' if year in [2017, 2018] else 'hybrid2019'}_source_count"]
    present_count = row[f"sources_present_{year}"]

    return source_count > 0 and present_count == source_count


# Employment and mean wage:
# one-to-one or complete multiple-source merger
for year in [2017, 2018]:
    soc_correspondence_audit[f"eligible_emp_mean_{year}"] = (
        soc_correspondence_audit["soc2010_topology"]
        .isin(aggregable_topologies)
        &
        soc_correspondence_audit.apply(
            lambda row: all_sources_present(row, year),
            axis=1
        )
    )

soc_correspondence_audit["eligible_emp_mean_2019"] = (
    soc_correspondence_audit["hybrid2019_topology"]
    .isin(aggregable_topologies)
    &
    soc_correspondence_audit.apply(
        lambda row: all_sources_present(row, 2019),
        axis=1
    )
)


# Median wage requires one-to-one correspondence in every year
for year in [2017, 2018]:
    soc_correspondence_audit[f"eligible_median_{year}"] = (
        soc_correspondence_audit["soc2010_topology"].eq("one_to_one")
        &
        soc_correspondence_audit.apply(
            lambda row: all_sources_present(row, year),
            axis=1
        )
    )

soc_correspondence_audit["eligible_median_2019"] = (
    soc_correspondence_audit["hybrid2019_topology"].eq("one_to_one")
    &
    soc_correspondence_audit.apply(
        lambda row: all_sources_present(row, 2019),
        axis=1
    )
)


# Complete three-year structural eligibility
soc_correspondence_audit["eligible_emp_mean_pre"] = (
    soc_correspondence_audit[
        [
            "eligible_emp_mean_2017",
            "eligible_emp_mean_2018",
            "eligible_emp_mean_2019"
        ]
    ].all(axis=1)
)

soc_correspondence_audit["eligible_median_pre"] = (
    soc_correspondence_audit[
        [
            "eligible_median_2017",
            "eligible_median_2018",
            "eligible_median_2019"
        ]
    ].all(axis=1)
)


structural_eligibility_summary = pd.DataFrame({
    "outcome_family": [
        "Employment / mean wage",
        "Median wage"
    ],
    "structurally_eligible": [
        int(soc_correspondence_audit["eligible_emp_mean_pre"].sum()),
        int(soc_correspondence_audit["eligible_median_pre"].sum())
    ],
    "structurally_ineligible": [
        int((~soc_correspondence_audit["eligible_emp_mean_pre"]).sum()),
        int((~soc_correspondence_audit["eligible_median_pre"]).sum())
    ]
})

display(structural_eligibility_summary)

In [ ]:
# ------------------------------------------------------------
# Validate the four aggregable 2010→2018 mergers
# ------------------------------------------------------------

merger_targets = soc_correspondence_audit.loc[
    soc_correspondence_audit["soc2010_topology"]
    .eq("multiple_sources_to_one_target"),
    "soc6"
]

merger_review = (
    soc_crosswalk.loc[
        soc_crosswalk["soc2018"].isin(merger_targets),
        [
            "soc2010",
            "title2010",
            "soc2018",
            "title2018"
        ]
    ]
    .drop_duplicates()
    .merge(
        soc_correspondence_audit[
            [
                "soc6",
                "sources_present_2017",
                "sources_present_2018",
                "soc2010_source_count",
                "hybrid2019_topology",
                "sources_present_2019"
            ]
        ],
        left_on="soc2018",
        right_on="soc6",
        how="left",
        validate="many_to_one"
    )
    .drop(columns="soc6")
    .sort_values(["soc2018", "soc2010"])
    .reset_index(drop=True)
)

# Each historical source must belong only to one current target
assert (
    merger_review.groupby("soc2010")["soc2018"]
    .nunique()
    .eq(1)
    .all()
)

# Every required historical component must be available
merger_status = (
    merger_review.groupby(["soc2018", "title2018"], as_index=False)
    .agg(
        predecessor_count=("soc2010", "nunique"),
        present_2017=("sources_present_2017", "first"),
        present_2018=("sources_present_2018", "first"),
        hybrid2019_topology=("hybrid2019_topology", "first"),
        present_2019=("sources_present_2019", "first")
    )
)

assert len(merger_status) == 4
assert (
    merger_status["predecessor_count"]
    .eq(merger_status["present_2017"])
    .all()
)
assert (
    merger_status["predecessor_count"]
    .eq(merger_status["present_2018"])
    .all()
)

print("Four employment/mean-wage mergers validated.")
display(merger_review)
display(merger_status)

In [ ]:
# ------------------------------------------------------------
# Check outcome-field coverage within structurally eligible SOCs
# ------------------------------------------------------------

import numpy as np
import re

def resolve_column(columns, candidates):
    normalized = {
        re.sub(r"[^A-Z0-9]", "", str(col).upper()): col
        for col in columns
    }

    for candidate in candidates:
        key = re.sub(r"[^A-Z0-9]", "", candidate.upper())
        if key in normalized:
            return normalized[key]

    return None


def parse_oews_numeric(series):
    # "*" = unavailable; "#" = top-coded, so neither is treated as numeric
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "*": np.nan,
            "#": np.nan
        })
    )
    return pd.to_numeric(cleaned, errors="coerce")


# Standardize the six cached annual source tables
outcome_source_frames = {}

for year, frame in cached_frames.items():
    soc_col = resolve_column(
        frame.columns, ["soc6", "occ_code", "soc_code"]
    )
    emp_col = resolve_column(
        frame.columns, ["employment", "tot_emp"]
    )
    mean_col = resolve_column(
        frame.columns, ["annual_mean", "a_mean", "mean_wage"]
    )
    median_col = resolve_column(
        frame.columns, ["annual_median", "a_median", "median_wage"]
    )

    if None in [soc_col, emp_col, mean_col, median_col]:
        raise KeyError(
            f"{year}: required outcome columns were not found.\n"
            f"Columns: {frame.columns.tolist()}"
        )

    standardized = pd.DataFrame({
        "source_soc": frame[soc_col].map(extract_soc),
        "employment": parse_oews_numeric(frame[emp_col]),
        "annual_mean": parse_oews_numeric(frame[mean_col]),
        "annual_median": parse_oews_numeric(frame[median_col])
    }).dropna(subset=["source_soc"])

    assert standardized["source_soc"].is_unique, (
        f"{year}: duplicate source SOC codes detected"
    )

    outcome_source_frames[year] = standardized.set_index("source_soc")


emp_mean_targets = set(
    soc_correspondence_audit.loc[
        soc_correspondence_audit["eligible_emp_mean_pre"],
        "soc6"
    ]
)

median_targets = set(
    soc_correspondence_audit.loc[
        soc_correspondence_audit["eligible_median_pre"],
        "soc6"
    ]
)


coverage_rows = []

for _, row in soc_correspondence_audit.iterrows():
    target = row["soc6"]

    for year in analysis_years:
        if year in [2017, 2018]:
            sources = row["soc2010_source_codes"].split(" | ")
        elif year == 2019:
            sources = row["hybrid2019_source_codes"].split(" | ")
        else:
            sources = [target]

        sources = [code for code in sources if code]
        available = outcome_source_frames[year].reindex(sources)

        coverage_rows.append({
            "soc6": target,
            "year": year,
            "source_count": len(sources),
            "employment_available": (
                len(sources) > 0
                and available["employment"].notna().all()
                and available["employment"].gt(0).all()
            ),
            "annual_mean_available": (
                len(sources) > 0
                and available["annual_mean"].notna().all()
            ),
            "annual_median_available": (
                len(sources) == 1
                and available["annual_median"].notna().all()
            )
        })


outcome_coverage = pd.DataFrame(coverage_rows)

coverage_summary = []

for year in analysis_years:
    year_data = outcome_coverage[
        outcome_coverage["year"].eq(year)
    ]

    coverage_summary.append({
        "year": year,
        "employment_available": year_data.loc[
            year_data["soc6"].isin(emp_mean_targets),
            "employment_available"
        ].sum(),
        "employment_denominator": len(emp_mean_targets),

        "annual_mean_available": year_data.loc[
            year_data["soc6"].isin(emp_mean_targets),
            "annual_mean_available"
        ].sum(),
        "annual_mean_denominator": len(emp_mean_targets),

        "annual_median_available": year_data.loc[
            year_data["soc6"].isin(median_targets),
            "annual_median_available"
        ].sum(),
        "annual_median_denominator": len(median_targets)
    })


outcome_coverage_summary = pd.DataFrame(coverage_summary)
display(outcome_coverage_summary)

In [ ]:
# ------------------------------------------------------------
# Determine complete six-year outcome samples
# ------------------------------------------------------------

complete_coverage = (
    outcome_coverage.groupby("soc6", as_index=False)
    .agg(
        employment_complete=(
            "employment_available", "all"
        ),
        annual_mean_complete=(
            "annual_mean_available", "all"
        ),
        annual_median_complete=(
            "annual_median_available", "all"
        )
    )
)

complete_coverage["employment_sample"] = (
    complete_coverage["soc6"].isin(emp_mean_targets)
    & complete_coverage["employment_complete"]
)

complete_coverage["annual_mean_sample"] = (
    complete_coverage["soc6"].isin(emp_mean_targets)
    & complete_coverage["annual_mean_complete"]
)

complete_coverage["annual_median_sample"] = (
    complete_coverage["soc6"].isin(median_targets)
    & complete_coverage["annual_median_complete"]
)


final_coverage_summary = pd.DataFrame({
    "outcome": [
        "Employment",
        "Annual mean wage",
        "Annual median wage"
    ],
    "structural_maximum": [
        len(emp_mean_targets),
        len(emp_mean_targets),
        len(median_targets)
    ],
    "complete_six_year_sample": [
        int(complete_coverage["employment_sample"].sum()),
        int(complete_coverage["annual_mean_sample"].sum()),
        int(complete_coverage["annual_median_sample"].sum())
    ]
})

display(final_coverage_summary)


# Show exactly which SOC-year wage observations are unavailable
missing_wage_observations = outcome_coverage.loc[
    (
        outcome_coverage["soc6"].isin(emp_mean_targets)
        & ~outcome_coverage["annual_mean_available"]
    )
    |
    (
        outcome_coverage["soc6"].isin(median_targets)
        & ~outcome_coverage["annual_median_available"]
    ),
    [
        "soc6",
        "year",
        "annual_mean_available",
        "annual_median_available"
    ]
].sort_values(["soc6", "year"])

display(missing_wage_observations)

In [ ]:
# ------------------------------------------------------------
# Diagnose why wage observations are unavailable
# ------------------------------------------------------------

def classify_bls_token(value):
    token = str(value).strip()

    if token == "*":
        return "unavailable"
    if token == "#":
        return "top_coded"
    if token == "" or token.lower() == "nan":
        return "blank"
    if pd.to_numeric(token.replace(",", ""), errors="coerce") is not np.nan:
        return "numeric"

    return "other_nonnumeric"


raw_wage_frames = {}

for year, frame in cached_frames.items():
    soc_col = resolve_column(
        frame.columns, ["soc6", "occ_code", "soc_code"]
    )
    title_col = resolve_column(
        frame.columns,
        ["occupation_title", "occ_title", "occupation_title_bls"]
    )
    mean_col = resolve_column(
        frame.columns, ["annual_mean", "a_mean", "mean_wage"]
    )
    median_col = resolve_column(
        frame.columns, ["annual_median", "a_median", "median_wage"]
    )

    raw_frame = pd.DataFrame({
        "source_soc": frame[soc_col].map(extract_soc),
        "source_title": (
            frame[title_col].astype(str).str.strip()
            if title_col is not None
            else pd.NA
        ),
        "annual_mean_raw": frame[mean_col],
        "annual_median_raw": frame[median_col]
    }).dropna(subset=["source_soc"])

    raw_wage_frames[year] = raw_frame.set_index("source_soc")


audit_by_soc = soc_correspondence_audit.set_index("soc6")
diagnostic_rows = []

for _, missing_row in missing_wage_observations.iterrows():
    target = missing_row["soc6"]
    year = missing_row["year"]
    audit_row = audit_by_soc.loc[target]

    if year in [2017, 2018]:
        sources = audit_row["soc2010_source_codes"].split(" | ")
    elif year == 2019:
        sources = audit_row["hybrid2019_source_codes"].split(" | ")
    else:
        sources = [target]

    for source in [code for code in sources if code]:
        source_record = raw_wage_frames[year].loc[source]

        if not missing_row["annual_mean_available"]:
            diagnostic_rows.append({
                "soc6": target,
                "year": year,
                "source_soc": source,
                "source_title": source_record["source_title"],
                "wage_field": "annual_mean",
                "raw_token": source_record["annual_mean_raw"],
                "reason": classify_bls_token(
                    source_record["annual_mean_raw"]
                )
            })

        if not missing_row["annual_median_available"]:
            diagnostic_rows.append({
                "soc6": target,
                "year": year,
                "source_soc": source,
                "source_title": source_record["source_title"],
                "wage_field": "annual_median",
                "raw_token": source_record["annual_median_raw"],
                "reason": classify_bls_token(
                    source_record["annual_median_raw"]
                )
            })


wage_missingness_diagnostic = pd.DataFrame(diagnostic_rows)

display(
    wage_missingness_diagnostic.groupby(
        ["wage_field", "reason"]
    ).agg(
        affected_occupations=("soc6", "nunique"),
        affected_observations=("soc6", "size")
    ).reset_index()
)

display(
    wage_missingness_diagnostic.sort_values(
        ["wage_field", "soc6", "year"]
    )
)

Employment outcomes will use 541 occupations. Primary wage outcomes will use the 538 occupations with complete annual mean wages, while annual median-wage models will be reported as a 527-occupation robustness analysis.

Primary mean-wage model: N = 538.

Median-wage robustness model: N = 527.

Mean-wage model rerun on the same N = 527 occupations.

**Main manuscript:** employment model (\(N=541\)) and mean-wage model (\(N=538\)).

**Supplement:** one compact robustness table containing:

median-wage model (\(N=527\));

mean-wage model on the identical \(N=527\) sample.

**Main text:** one sentence stating whether conclusions remain consistent across the median and common-sample checks.

In [ ]:
# ------------------------------------------------------------
# Construct harmonized annual employment and nominal wage panel
# ------------------------------------------------------------

audit_lookup = soc_correspondence_audit.set_index("soc6")
annual_rows = []

for soc6 in sorted(target_socs):
    audit_row = audit_lookup.loc[soc6]

    for year in analysis_years:

        if year in [2017, 2018]:
            source_codes = [
                code for code
                in audit_row["soc2010_source_codes"].split(" | ")
                if code
            ]
            mapping_method = audit_row["soc2010_topology"]
            emp_mean_ok = audit_row[f"eligible_emp_mean_{year}"]
            median_ok = audit_row[f"eligible_median_{year}"]

        elif year == 2019:
            source_codes = [
                code for code
                in audit_row["hybrid2019_source_codes"].split(" | ")
                if code
            ]
            mapping_method = audit_row["hybrid2019_topology"]
            emp_mean_ok = audit_row["eligible_emp_mean_2019"]
            median_ok = audit_row["eligible_median_2019"]

        else:
            source_codes = [soc6]
            mapping_method = "direct_2018_soc"
            emp_mean_ok = True
            median_ok = True

        source_values = outcome_source_frames[year].reindex(
            source_codes
        )

        employment = np.nan
        annual_mean = np.nan
        annual_median = np.nan

        # Employment is additive across valid predecessor occupations
        if (
            emp_mean_ok
            and len(source_codes) > 0
            and source_values["employment"].notna().all()
            and source_values["employment"].gt(0).all()
        ):
            employment = source_values["employment"].sum()

            # Combined mean = employment-weighted component mean
            if source_values["annual_mean"].notna().all():
                annual_mean = np.average(
                    source_values["annual_mean"],
                    weights=source_values["employment"]
                )

        # Median is retained only for a single, valid source occupation
        if (
            median_ok
            and len(source_codes) == 1
            and source_values["annual_median"].notna().all()
        ):
            annual_median = source_values[
                "annual_median"
            ].iloc[0]

        annual_rows.append({
            "soc6": soc6,
            "year": year,
            "mapping_method": mapping_method,
            "source_codes": " | ".join(source_codes),
            "source_count": len(source_codes),
            "employment": employment,
            "annual_mean_nominal": annual_mean,
            "annual_median_nominal": annual_median,
            "emp_mean_structural_ok": bool(emp_mean_ok),
            "median_structural_ok": bool(median_ok)
        })


annual_outcome_panel = pd.DataFrame(annual_rows)

assert len(annual_outcome_panel) == 604 * len(analysis_years)
assert not annual_outcome_panel.duplicated(["soc6", "year"]).any()


panel_coverage = (
    annual_outcome_panel.groupby("year", as_index=False)
    .agg(
        occupations=("soc6", "size"),
        employment_available=("employment", "count"),
        annual_mean_available=("annual_mean_nominal", "count"),
        annual_median_available=("annual_median_nominal", "count")
    )
)

display(panel_coverage)
display(annual_outcome_panel.head(12))

In [ ]:
# ------------------------------------------------------------
# Locked May CPI-U values for the six analysis years
# Series: CUUR0000SA0
# U.S. city average, all items, not seasonally adjusted
# ------------------------------------------------------------

cpi_may = pd.DataFrame({
    "year": [
        2017, 2018, 2019,
        2023, 2024, 2025
    ],
    "cpi_may": [
        244.733, 251.588, 256.092,
        304.127, 314.069, 321.465
    ]
})

assert set(cpi_may["year"]) == set(analysis_years)
assert cpi_may["year"].is_unique

CPI_BASE_YEAR = 2025
cpi_base = cpi_may.loc[
    cpi_may["year"].eq(CPI_BASE_YEAR),
    "cpi_may"
].iloc[0]


annual_outcome_panel = annual_outcome_panel.merge(
    cpi_may,
    on="year",
    how="left",
    validate="many_to_one"
)

assert annual_outcome_panel["cpi_may"].notna().all()

annual_outcome_panel["annual_mean_real_2025"] = (
    annual_outcome_panel["annual_mean_nominal"]
    * cpi_base
    / annual_outcome_panel["cpi_may"]
)

annual_outcome_panel["annual_median_real_2025"] = (
    annual_outcome_panel["annual_median_nominal"]
    * cpi_base
    / annual_outcome_panel["cpi_may"]
)

display(cpi_may)

display(
    annual_outcome_panel[
        [
            "soc6",
            "year",
            "employment",
            "annual_mean_nominal",
            "annual_mean_real_2025",
            "annual_median_nominal",
            "annual_median_real_2025"
        ]
    ].head(12)
)

In [ ]:
# ------------------------------------------------------------
# Construct pre/post averages and annualized log changes
# ------------------------------------------------------------

PRE_YEARS = [2017, 2018, 2019]
POST_YEARS = [2023, 2024, 2025]

PRE_MIDPOINT = np.mean(PRE_YEARS)      # 2018
POST_MIDPOINT = np.mean(POST_YEARS)    # 2024
YEAR_GAP = POST_MIDPOINT - PRE_MIDPOINT

assert YEAR_GAP == 6


pre_endpoints = (
    annual_outcome_panel[
        annual_outcome_panel["year"].isin(PRE_YEARS)
    ]
    .groupby("soc6", as_index=False)
    .agg(
        employment_pre_avg=("employment", "mean"),
        mean_wage_real_pre_avg=("annual_mean_real_2025", "mean"),
        median_wage_real_pre_avg=("annual_median_real_2025", "mean")
    )
)

post_endpoints = (
    annual_outcome_panel[
        annual_outcome_panel["year"].isin(POST_YEARS)
    ]
    .groupby("soc6", as_index=False)
    .agg(
        employment_post_avg=("employment", "mean"),
        mean_wage_real_post_avg=("annual_mean_real_2025", "mean"),
        median_wage_real_post_avg=("annual_median_real_2025", "mean")
    )
)


occupational_outcomes = (
    pre_endpoints
    .merge(
        post_endpoints,
        on="soc6",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        complete_coverage[
            [
                "soc6",
                "employment_sample",
                "annual_mean_sample",
                "annual_median_sample"
            ]
        ],
        on="soc6",
        how="left",
        validate="one_to_one"
    )
)

assert len(occupational_outcomes) == 604
assert occupational_outcomes["soc6"].is_unique


def annualized_log_change(pre, post, eligible):
    valid = (
        eligible
        & pre.notna()
        & post.notna()
        & pre.gt(0)
        & post.gt(0)
    )

    result = pd.Series(np.nan, index=pre.index)

    result.loc[valid] = (
        np.log(post.loc[valid])
        - np.log(pre.loc[valid])
    ) / YEAR_GAP

    return result


occupational_outcomes["employment_ann_log_change"] = (
    annualized_log_change(
        occupational_outcomes["employment_pre_avg"],
        occupational_outcomes["employment_post_avg"],
        occupational_outcomes["employment_sample"]
    )
)

occupational_outcomes["mean_wage_ann_log_change"] = (
    annualized_log_change(
        occupational_outcomes["mean_wage_real_pre_avg"],
        occupational_outcomes["mean_wage_real_post_avg"],
        occupational_outcomes["annual_mean_sample"]
    )
)

occupational_outcomes["median_wage_ann_log_change"] = (
    annualized_log_change(
        occupational_outcomes["median_wage_real_pre_avg"],
        occupational_outcomes["median_wage_real_post_avg"],
        occupational_outcomes["annual_median_sample"]
    )
)


# Percentage-point presentation versions
occupational_outcomes["employment_ann_log_pct"] = (
    100 * occupational_outcomes["employment_ann_log_change"]
)

occupational_outcomes["mean_wage_ann_log_pct"] = (
    100 * occupational_outcomes["mean_wage_ann_log_change"]
)

occupational_outcomes["median_wage_ann_log_pct"] = (
    100 * occupational_outcomes["median_wage_ann_log_change"]
)


# Validate the frozen analysis samples
assert occupational_outcomes["employment_ann_log_change"].count() == 541
assert occupational_outcomes["mean_wage_ann_log_change"].count() == 538
assert occupational_outcomes["median_wage_ann_log_change"].count() == 527

print("Annualization interval:", YEAR_GAP, "years")

display(
    occupational_outcomes[
        [
            "employment_ann_log_pct",
            "mean_wage_ann_log_pct",
            "median_wage_ann_log_pct"
        ]
    ]
    .describe()
    .T
)

In [ ]:
# ------------------------------------------------------------
# Audit extreme occupational outcome changes
# ------------------------------------------------------------

occupation_titles = (
    raw_wage_frames[2025]
    .reset_index()[["source_soc", "source_title"]]
    .rename(columns={
        "source_soc": "soc6",
        "source_title": "occupation_title"
    })
    .drop_duplicates("soc6")
)

outcome_audit = (
    occupational_outcomes
    .merge(
        occupation_titles,
        on="soc6",
        how="left",
        validate="one_to_one"
    )
    .merge(
        soc_correspondence_audit[
            [
                "soc6",
                "soc2010_topology",
                "soc2010_source_codes",
                "hybrid2019_topology"
            ]
        ],
        on="soc6",
        how="left",
        validate="one_to_one"
    )
)

outcome_specs = {
    "Employment": {
        "change": "employment_ann_log_pct",
        "pre": "employment_pre_avg",
        "post": "employment_post_avg"
    },
    "Annual mean wage": {
        "change": "mean_wage_ann_log_pct",
        "pre": "mean_wage_real_pre_avg",
        "post": "mean_wage_real_post_avg"
    },
    "Annual median wage": {
        "change": "median_wage_ann_log_pct",
        "pre": "median_wage_real_pre_avg",
        "post": "median_wage_real_post_avg"
    }
}

extreme_tables = []

for outcome_name, columns in outcome_specs.items():
    valid = outcome_audit.dropna(subset=[columns["change"]])

    lowest = valid.nsmallest(5, columns["change"]).copy()
    lowest["tail"] = "Lowest"

    highest = valid.nlargest(5, columns["change"]).copy()
    highest["tail"] = "Highest"

    extremes = pd.concat(
        [lowest, highest],
        ignore_index=True
    )

    extremes["outcome"] = outcome_name
    extremes["pre_value"] = extremes[columns["pre"]]
    extremes["post_value"] = extremes[columns["post"]]
    extremes["annualized_log_pct"] = extremes[columns["change"]]

    extreme_tables.append(
        extremes[
            [
                "outcome",
                "tail",
                "soc6",
                "occupation_title",
                "pre_value",
                "post_value",
                "annualized_log_pct",
                "soc2010_topology",
                "soc2010_source_codes",
                "hybrid2019_topology"
            ]
        ]
    )

extreme_outcome_audit = pd.concat(
    extreme_tables,
    ignore_index=True
)

display(extreme_outcome_audit)

In [ ]:
# ------------------------------------------------------------
# Inspect annual trajectories behind extreme wage changes
# ------------------------------------------------------------

mean_valid = outcome_audit.dropna(
    subset=["mean_wage_ann_log_pct"]
)

median_valid = outcome_audit.dropna(
    subset=["median_wage_ann_log_pct"]
)

mean_extremes = pd.concat([
    mean_valid.nsmallest(3, "mean_wage_ann_log_pct"),
    mean_valid.nlargest(3, "mean_wage_ann_log_pct")
])

median_extremes = pd.concat([
    median_valid.nsmallest(3, "median_wage_ann_log_pct"),
    median_valid.nlargest(3, "median_wage_ann_log_pct")
])

wage_extreme_summary = pd.concat([
    mean_extremes.assign(wage_measure="Annual mean"),
    median_extremes.assign(wage_measure="Annual median")
])[
    [
        "wage_measure",
        "soc6",
        "occupation_title",
        "mean_wage_ann_log_pct",
        "median_wage_ann_log_pct",
        "soc2010_topology",
        "hybrid2019_topology"
    ]
].drop_duplicates()

display(wage_extreme_summary)


extreme_soc_codes = wage_extreme_summary["soc6"].unique()

wage_trajectories = (
    annual_outcome_panel[
        annual_outcome_panel["soc6"].isin(extreme_soc_codes)
    ]
    .merge(
        occupation_titles,
        on="soc6",
        how="left",
        validate="many_to_one"
    )
)

display(
    wage_trajectories.pivot(
        index=["soc6", "occupation_title"],
        columns="year",
        values="annual_mean_real_2025"
    )
)

display(
    wage_trajectories.pivot(
        index=["soc6", "occupation_title"],
        columns="year",
        values="annual_median_real_2025"
    )
)

In [ ]:
outcome_dir = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Output"
)

outcome_dir.mkdir(parents=True, exist_ok=True)

annual_outcome_panel.to_csv(
    outcome_dir / "OEWS_Annual_Harmonized_Panel_v1.csv",
    index=False
)

occupational_outcomes.to_csv(
    outcome_dir / "OEWS_Occupational_Outcomes_v1.csv",
    index=False
)

In [ ]:
save_check = {
    "annual_panel_rows": len(annual_outcome_panel),
    "annual_panel_occupations": annual_outcome_panel["soc6"].nunique(),
    "outcome_rows": len(occupational_outcomes),
    "employment_sample": occupational_outcomes[
        "employment_ann_log_change"
    ].notna().sum(),
    "mean_wage_sample": occupational_outcomes[
        "mean_wage_ann_log_change"
    ].notna().sum(),
    "median_wage_sample": occupational_outcomes[
        "median_wage_ann_log_change"
    ].notna().sum(),
}

print(f"Files saved to: {outcome_dir}")
save_check

In [ ]:
sample_year = analysis_years[0]

print("Year:", sample_year)
print("Columns:")
print(outcome_source_frames[sample_year].columns.tolist())

display(outcome_source_frames[sample_year].head())

In [ ]:
national_total_check = []

for year in analysis_years:
    frame = outcome_source_frames[year]

    total_found = "00-0000" in frame.index

    national_total_check.append({
        "year": year,
        "total_row_found": total_found,
        "national_employment": (
            frame.loc["00-0000", "employment"]
            if total_found
            else None
        )
    })

national_total_check = pd.DataFrame(national_total_check)

display(national_total_check)

In [ ]:
configuration_file = (
    BASE_DIR
    / "07_GenAI_Exposure"
    / "Output"
    / "Occupational_Configuration_Exposure_v1.csv"
)

outcome_file = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Output"
    / "OEWS_Occupational_Outcomes_v1.csv"
)

configuration_data = pd.read_csv(
    configuration_file,
    dtype={"soc6": "string"}
)

outcome_data = pd.read_csv(
    outcome_file,
    dtype={"soc6": "string"}
)

configuration_data["soc6"] = configuration_data["soc6"].str.strip()
outcome_data["soc6"] = outcome_data["soc6"].str.strip()

assert configuration_data["soc6"].is_unique
assert outcome_data["soc6"].is_unique

merge_audit = (
    configuration_data[["soc6"]]
    .merge(
        outcome_data[["soc6"]],
        on="soc6",
        how="outer",
        indicator=True,
        validate="one_to_one"
    )
)

display(
    merge_audit["_merge"]
    .value_counts()
    .rename_axis("merge_status")
    .reset_index(name="occupation_count")
)

In [ ]:
analysis_data = configuration_data.merge(
    outcome_data,
    on="soc6",
    how="left",
    validate="one_to_one",
    suffixes=("_configuration", "_outcome")
)

analysis_data["sample_employment"] = (
    analysis_data["employment_ann_log_change"].notna()
)

analysis_data["sample_mean_wage"] = (
    analysis_data["mean_wage_ann_log_change"].notna()
)

analysis_data["sample_median_wage"] = (
    analysis_data["median_wage_ann_log_change"].notna()
)

analysis_data["sample_common_wage"] = (
    analysis_data["sample_mean_wage"]
    & analysis_data["sample_median_wage"]
)

sample_audit = pd.DataFrame({
    "sample": [
        "Frozen occupation universe",
        "Employment",
        "Annual mean wage",
        "Annual median wage",
        "Common wage sample"
    ],
    "occupation_count": [
        len(analysis_data),
        analysis_data["sample_employment"].sum(),
        analysis_data["sample_mean_wage"].sum(),
        analysis_data["sample_median_wage"].sum(),
        analysis_data["sample_common_wage"].sum()
    ]
})

assert len(analysis_data) == 604

display(sample_audit)

In [ ]:
configuration_schema = pd.DataFrame({
    "variable": configuration_data.columns,
    "dtype": configuration_data.dtypes.astype(str).values,
    "missing_count": configuration_data.isna().sum().values,
    "unique_values": configuration_data.nunique(dropna=True).values
})

display(configuration_schema)

In [ ]:
layer_validation = pd.DataFrame({
    "check": [
        "Layer 1 shares sum to one",
        "Layer 2 shares sum to one",
        "Automation position within [0,1]",
        "Augmentation position within [0,1]",
        "Limited position within [0,1]"
    ],
    "passed": [
        np.allclose(
            configuration_data[
                ["automation_share", "augmentation_share", "limited_share"]
            ].sum(axis=1),
            1.0
        ),
        np.allclose(
            configuration_data[
                [
                    "automation_core_weighted_share",
                    "augmentation_core_weighted_share",
                    "limited_core_weighted_share"
                ]
            ].sum(axis=1),
            1.0
        ),
        configuration_data["automation_position"].dropna().between(0, 1).all(),
        configuration_data["augmentation_position"].dropna().between(0, 1).all(),
        configuration_data["limited_position"].dropna().between(0, 1).all()
    ]
})

sector_counts = pd.DataFrame({
    "sector": ["Finance", "Healthcare", "Education", "Legal"],
    "occupation_count": [
        configuration_data["sector_finance"].sum(),
        configuration_data["sector_healthcare"].sum(),
        configuration_data["sector_education"].sum(),
        configuration_data["sector_legal"].sum()
    ]
})

display(layer_validation)
display(sector_counts)

In [ ]:
sector_columns = [
    "sector_finance",
    "sector_healthcare",
    "sector_education",
    "sector_legal"
]

analysis_data["sector_membership_count"] = (
    analysis_data[sector_columns].astype(int).sum(axis=1)
)

membership_count_audit = (
    analysis_data["sector_membership_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_sectors")
    .reset_index(name="occupation_count")
)

sector_combination_audit = (
    analysis_data
    .groupby(sector_columns, dropna=False)
    .size()
    .reset_index(name="occupation_count")
    .sort_values("occupation_count", ascending=False)
    .reset_index(drop=True)
)

print(
    "Total sector–occupation memberships:",
    int(analysis_data[sector_columns].sum().sum())
)

display(membership_count_audit)
display(sector_combination_audit)

In [ ]:
sector_sample_audit = []

for sector_column in sector_columns:
    sector_name = sector_column.replace("sector_", "").title()
    sector_mask = analysis_data[sector_column]

    sector_sample_audit.append({
        "sector": sector_name,
        "configuration_occupations": int(sector_mask.sum()),
        "employment_sample": int(
            (sector_mask & analysis_data["sample_employment"]).sum()
        ),
        "mean_wage_sample": int(
            (sector_mask & analysis_data["sample_mean_wage"]).sum()
        ),
        "median_wage_sample": int(
            (sector_mask & analysis_data["sample_median_wage"]).sum()
        )
    })

sector_sample_audit = pd.DataFrame(sector_sample_audit)

display(sector_sample_audit)

In [ ]:
exposure_validation = pd.DataFrame({
    "exposure_source": ["Human", "GPT-4"],
    "maximum_beta_difference": [
        (
            configuration_data["human_beta"]
            - (
                configuration_data["human_alpha"]
                + configuration_data["human_gamma"]
            ) / 2
        ).abs().max(),
        (
            configuration_data["gpt4_beta"]
            - (
                configuration_data["gpt4_alpha"]
                + configuration_data["gpt4_gamma"]
            ) / 2
        ).abs().max()
    ]
})

mechanism_presence_audit = pd.DataFrame({
    "mechanism": ["Automation", "Augmentation", "Limited"],
    "present": [
        (configuration_data["automation_count"] > 0).sum(),
        (configuration_data["augmentation_count"] > 0).sum(),
        (configuration_data["limited_count"] > 0).sum()
    ],
    "absent": [
        (configuration_data["automation_count"] == 0).sum(),
        (configuration_data["augmentation_count"] == 0).sum(),
        (configuration_data["limited_count"] == 0).sum()
    ]
})

display(exposure_validation)
display(mechanism_presence_audit)

In [ ]:
unexpected_position_missing = (
    ((configuration_data["automation_count"] > 0)
     & configuration_data["automation_position"].isna())
    |
    ((configuration_data["augmentation_count"] > 0)
     & configuration_data["augmentation_position"].isna())
    |
    ((configuration_data["limited_count"] > 0)
     & configuration_data["limited_position"].isna())
)

position_gap_socs = configuration_data.loc[
    unexpected_position_missing,
    [
        "soc6",
        "automation_count",
        "automation_position",
        "augmentation_count",
        "augmentation_position",
        "limited_count",
        "limited_position"
    ]
].copy()

print(
    "Unique occupations with positive counts but missing positions:",
    position_gap_socs["soc6"].nunique()
)

display(position_gap_socs)